# icepyx and ICESat-2 data in the cloud  
----------- 
<figure>
<left>
    <img src="https://icepyx.readthedocs.io/en/latest/_static/icepyx_v2_oval_orig_nobackgr.png" alt='icepyx logo of the word icepyx in raised letters on an iceberg with an ice ax' style='width: 200px;'/>
    <img src="https://icesat-2.gsfc.nasa.gov/themes/custom/icesat2/images/MissionLogo.png" alt='ICESat-2 mission logo of the word ICESat-2 with penguin making the shape of the two' style='width: 250px;'/>
</left>
</figure>

-----------
**Presented at**: [Building Open Connected Scientific Data Products for the Cryosphere hackdays](https://englacial.org/pages/2026-hackdays.html)  
13-15 April 2026  
Berkeley, CA, USA  
**Created and presented by**: Jessica Scheick  
**Credits** Includes content from tutorials and presentations by Rachel Wegener (Univ. Maryland), Jessica Scheick (Univ. New Hampshire), and Amy Steiker (NSIDC)

## Objectives

- introduce icepyx as a software package and community
- provide context for ICESat-2 data in the cloud
- demonstrate how to query, access, and plot ICESat-2 data in the cloud using `icepyx`

### Reference Info

- [Awesome ICESat-2](https://github.com/ICESat-2/awesome-icesat2): gently opinionated collated collection of ICESat-2 resources
- [ICESat-2 Cookbook](https://icesat-2.github.io/icesat2-cookbook/): Jupyter Book of tutorials collated and updated from across six Hackweek events (note: updates to some tutorials are still in progress)
- [Mission Overview](https://icesat-2.github.io/icesat2-cookbook/notebooks/mission-overview/): learn more about the mission, ATLAS sensor, and available data products

## icepyx origin story and brief history

icepyx is a community and Python software library for searching, downloading, and reading ICESat-2 data. ICESat-2 data has a highly nested structure and - before tools like `earthaccess` existed - required > 50 lines of code to get programmatically. `icepyx` provides tools to ease these data discovery, access, and complexity challenges.

![about_icepyx](IS2-in-cloud_images/about-ipx.png)

`icepyx` has seen wide adoption across the ICESat-2 data user and developer communities.

![icepyx_statistics](IS2-in-cloud_images/icepyx-stats.png)

Thanks to contributions from countless community members, `icepyx` can (for ICESat-2 data): 
- search for available data granules (data files)
- order and download data or access it directly in the cloud
- order a subset of data: clipped in space, time, or a few other options provided by NSIDC
- navigate the available ICESat-2 data variables
- read ICESat-2 data into Xarray DataArrays, including merging data from multiple files
- access coincident Argo data via the QUEST (Query Unify Explore SpatioTemporal) module
- add new datasets to QUEST via a template

Under the hood, `icepyx` relies on `earthaccess` to help handle authentication, especially for obtaining S3 tokens to access ICESat-2 data in the cloud. All this happens without the user needing to take any action other than supplying their Earthdata Login credentials using one of the methods described in the `earthaccess` tutorial.

## ICESat-2 data in the cloud

The tale of ICESat-2 data in the cloud - and critically for data users, the accessibility and usability of that data in the cloud - is ongoing. It includes stories of collaboration, celebration of successful milestones, and many opportunities for continued exploration and improvement.

### 🗓️ ICESat-2 Cloud Access Timeline

| Date | Milestone | Details |
|---|---|---|
| Sept 2022 | ☁️ First Cloud Access | ICESat-2 data becomes publicly available on the NASA Earthdata Cloud (AWS). |
| October 2022 | ✨ CryoCloud launched | CryoCloud Jupyter Hub provides persistent cloud access. |
| June 2023 |  🚀 h5coro Python Released | Python implementation of h5coro released. |
| May 2025 | 🚀 icepyx v2 Released | icepyx transitions backend to cloud-based Harmony API. |
| May 2025 | ☁️ Migration Milestone | Earthdata Cloud becomes the primary archive; legacy on-premise systems begin retirement. |
| Summer 2025 | 🚀 h5coro Xarray backend | h5coro available as an Xarray backend. |
| June 2025 | ☁️ Quick Look Shift | Version 7 "Quick Look" products (3-day latency) move exclusively to the cloud. |
| July 2025 | ☁️ Cloud-Optimized HDF5 | ATL03 (photon data) is released in cloud-optimized HDF5 format, enabling faster, partial-file reads. |
| Aug 2025 | ✨ ICESat-2 Developer's Hackweek | Project leads from across ICESat-2 ecosystem work to improve tools and documentation for working with ICESat-2 data in the cloud. |
| Sept 2025 | ☁️ Scaling Up | Additional Level-3A and 3B datasets are released as cloud-optimized HDF5. |

### Technicalities/challenges
- non-cloud-optimized HDF5 files meant impossibly slow streaming times
- cloud hub access was inconsistent

**Options for reading data in the cloud**
| Tool | Read Level | Notes |
|---|---|---|
| icepyx | group or variable (for all beams) | Creates an Xarray DataSet with all requested data combined; not the most performant |
| Xarray with h5coro engine | group (per beam) | Performant per group, but requires manual combining of groups (incurs more compute); cannot read groups containing variables of different lengths (e.g. orbit_info) |
| h5coro | variable (per beam) | Parallel reads; users must manage data within dictionary |
| SlideRule | variable | Not a cloud-reader; highly performant; not available for all products |
| Harmony | granule | No variable subsetting; if not streaming full granules, must wait for endpoint to process and serve subset granules |


### The Tooling Landscape
![ICESat-2 tooling landscape](IS2-in-cloud_images/ecosystem.png)

## icepyx Demo - reading ICESat-2 data in the cloud

In this tutorial we will look at the `ATL06` Land Ice Height product.

**Demo Objectives**
- use `icepyx` to search for ICESat-2 ATL06 Land Ice Height product for a given spatial and temporal extent
- select a series of variables of interest
- stream the requested data in the cloud into an `xarray.Dataset` using `icepyx.Read`
- plot the data as a first step towards further data analysis

**Demo Prerequisites**
- Jupyter Hub (e.g. CryoCloud) or AWS EC2 instance in the us-west-2 region for running the notebook in the cloud
- NASA Earthdata login for data access
- clone the icepyx repo

Under the hood, `icepyx` relies on `earthaccess` to help handle authentication, especially for obtaining S3 tokens to access ICESat-2 data in the cloud. All this happens without the user needing to take any action other than supplying their Earthdata Login credentials using one of the methods described in the [`earthaccess` documentation](https://earthaccess.readthedocs.io/en/stable/user/authenticate/).

In [ ]:
# note that we're using the cryo-hack branch of icepyx, as this code is not yet in the latest release
# if you haven't already, you'll need to git clone the repo into your JupyterHub and install locally
!git clone -b cryo-hack https://github.com/icesat2py/icepyx.git

In [ ]:
# install icepyx and restart your kernel
%pip install ~/icepyx

# NOTE: icepyx is a standard library in the CryoCloud Jupyter Hub default image, but we need this branch for this demo

In [ ]:
import datashader
import geoviews as gv
import hvplot.xarray
import matplotlib.pyplot as plt

# %load_ext autoreload
import icepyx as ipx
# %autoreload 2

In [ ]:
%matplotlib inline

### Define our search parameters

In [ ]:
# Use our search parameters to setup a search Query
short_name = 'ATL06'
spatial_extent = [-39, 66.2, -37.7, 66.6]
date_range = ['2019-05-04','2019-07-04'] # '2019-05-10' will get you one granule only
region = ipx.Query(short_name, spatial_extent, date_range)

In [ ]:
# Visualize our spatial extent
region.visualize_spatial_extent()

In [ ]:
# show granule IDs and S3 urls
region.avail_granules(ids=True, cloud=True)

In [ ]:
s3urls = region.avail_granules(ids=True, cloud=True)[1]

### Reading a file with icepyx

To read a file with icepyx there are several steps:
1. Create a `Read` object. This sets up an initial connection to your file(s) and validates the metadata.
2. Tell the `Read` object what variables you would like to read
3. Load your data!

#### Create the Read object

In [ ]:
# create the read object
reader = ipx.Read(s3urls)

<div class="alert alert-block alert-info">
<b>Tip:</b> If you don't want to type your Earthdata Login information every time they are
    required you can setup more automatic methods of authentication. Two common methods
    are 1) Add your earthdata password and username to as environment variables
    as EARTHDATA_USERNAME and EARTHDATA_PASSWORD. 2) setup a .netrc file in your home directory. See <a href="https://nasa-openscapes.github.io/2021-Cloud-Hackathon/tutorials/04_NASA_Earthdata_Authentication.html"> the Openscapes tutorial</a> </div>

In [ ]:
# view what files are associated with this Read object
reader.filelist

In [ ]:
# view the list of group paths+variables that are available for your data product
reader.variables.avail()

#### Background info on variables

That's **a lot** of variables!

One key feature of icepyx is the ability to browse the variables available in the dataset. There are typically hundreds of variables in a single dataset, so that is a lot to sort through! Let's take a moment to get oriented to the organization of ATL06 variables, starting with a few important pieces of the algorithm.

To create higher level variables like land ice height, the ATL06 algorithms goes through a series of steps:
1. Identify signal photons from noise photons
2. Filter out photons not over (or near) land
3. Group the signal photons into 40m segments. If there are a sufficient number of photons in that group, calculate statistics (ex. mean height, max height, standard deviation, etc.)

Providing all the potentially useful information from all these processing steps results in a data file that looks analogous to this diagram for an ATL08 file:

<img src="https://nasa-openscapes.github.io/2023-ssc/tutorials/data-access/.images/ATL08_structure.png" width=650/>

Another way to visualize these structure is to download one file and open it using https://myhdf5.hdfgroup.org/. 

Further information about each one of the variables is available in the [Algorithm Theoretical Basis Document (ATBD)](https://nsidc.org/sites/default/files/documents/technical-reference/icesat2_atl06_atbd_v007.pdf) for ATL06.

#### Select some variables

Here we will use the ATL06 variable `h_li`, land ice height. 

NOTE: adding variables is a required step before you can load the data.

See this [ICESat-2's Nested Variables](https://icepyx.readthedocs.io/en/latest/example_notebooks/IS2_data_variables.html) tutorial for more details on how to choose specific beams, variables, and keywords.

In [ ]:
# selected beams
reader.variables.append(beam_list=['gt2l','gt3l'], var_list=['h_li', 'latitude', 'longitude'])

In [ ]:
# show what variables we've requested (note some required ones are added during the load step)
reader.variables.wanted

#### Load the data!

In [ ]:
%%time
ds = reader.load()

In [ ]:
ds

### Visualizing our data

There are so many ways to do this - we're excited to see what you come up with!

(showcase some other approaches here?)

In [ ]:
tile = gv.tile_sources.EsriImagery.opts(width=500, height=500)

In [ ]:
# convert our data to geodetic coordinates and add them to the dataset
x, y = datashader.utils.lnglat_to_meters(ds.longitude, ds.latitude)

In [ ]:
ds = ds.assign(x=x, y=y)

In [ ]:
# compute x and y min and max for plotting
xmin, ymin = datashader.utils.lnglat_to_meters(spatial_extent[0], spatial_extent[1])
xmax, ymax = datashader.utils.lnglat_to_meters(spatial_extent[2], spatial_extent[3])

In [ ]:
# create our plot
is2 = ds.where(ds.h_li<3e38).hvplot.scatter(x="x", 
                        y="y", 
                        groupby=[], #"rgt"], 
                        xlim=(xmin, xmax),
                        ylim=(ymin, ymax),
                        color="h_li"
                       )

In [ ]:
is2

In [ ]:
# background via geoviews
is2 * tile

## Summary 

In this notebook we explored the opening and rendering ATL06 data with icepyx. We saw that icepyx will read in the desired variables directly in the cloud. The ATL06 data has a folder-like structure with many variables to choose from. We focused on `h_li`.

More information about ATL06 or icepyx can be found in:
- The [icepyx documentation](https://icepyx.readthedocs.io/en/latest/)
- The [Algorithm Theoretical Basis Document (ATBD)](https://icesat-2.gsfc.nasa.gov/sites/default/files/page_files/ICESat2_ATL06_ATBD_r007.pdf)